In [1]:
import pandas as pd
import time
import torch
from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

import pandas as pd
df = pd.read_csv("data_train/22U_1_3_4/mmwave_ss.csv")
df["kss_score"] = 4
df["log_time"] = pd.to_datetime(df['log_time'])
df = df.set_index("log_time")
df = df.resample("1s").mean().interpolate(method="linear")
df = df.reset_index()

target_ts = TimeSeries.from_dataframe(df, time_col="log_time", value_cols=["kss_score"])
past_cov_ts = TimeSeries.from_dataframe(df, time_col="log_time", value_cols=["breath_rate", "heart_rate"])

scaler_target = Scaler()
scaler_past_cov = Scaler()

target_scaled = scaler_target.fit_transform(target_ts)
past_cov_scaled = scaler_past_cov.fit_transform(past_cov_ts)

In [9]:
timestamp = time.time()
logger = CSVLogger(save_dir="logs/", name=f"tft_run_{timestamp}")

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1, # only save 1 file
    dirpath="logs/checkpoints/",
    filename=f"tft_{timestamp}_epoch{{epoch:02d}}_val{{val_loss:.4f}}",
    save_weights_only=False 
)

model = TFTModel(
    input_chunk_length=120, # change to seq_len * context_time
    output_chunk_length=30, # change to seq_len * prediction_time
    hidden_size=32,
    lstm_layers=2,
    num_attention_heads=4,
    dropout=0.1,
    batch_size=64, # change to batch_size
    n_epochs=5, # change to epoch
    optimizer_kwargs={"lr": 1e-3},
    loss_fn=torch.nn.MSELoss(),
    add_encoders={ # automate extract future_cov from timestamp/log_time
        'cyclic': {'future': ['minute', 'second', 'hour']},
        'transformer': Scaler()
    },
    
    pl_trainer_kwargs={ # trainer from pytorch lightning
        "accelerator": "auto",
        "callbacks": [checkpoint_callback],
        "logger": logger,
        "enable_checkpointing":True,
        "log_every_n_steps": 1
    },
    random_state=42, # seed so the experiment can be reproduced
)

In [10]:
model.fit(
    series=target_ts,
    past_covariates=past_cov_ts,
    val_series=target_ts,
    val_past_covariates=past_cov_ts,
    verbose=True,
    max_samples_per_ts=100 # get 1000 random windows for training
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/nuranisa/miniconda3/envs/ml_protel/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/nuranisa/Documents/PROTEL/ml/logs/checkpoints exists and is not empty.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                              ┃ Type                             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ criterion                         │ MSELoss                          │      0 │ train │     0 │
│ 1  │ train_criterion                   │ MSELoss                          │      0 │ train │     0 │
│ 2  │ val_criterion                     │ MSELoss                          │      0 │ train │     0 │
│ 3  │ train_metrics                     │ MetricCollection                 │      0 │ train │     0 │
│ 4  │ val_metrics                       │ MetricCollection                 │      0 │ train │     0 │
│ 5  │ input_embeddings                  │ _MultiEmbedding                  │      0 │ train │     0 │
│ 6  │ static_covariates_vsn             │ _VariableSelectionNetwork        │      0 │ train │     0 │
│ 7  │ encoder_vsn                       │ _VariableSelectionNetwork        │  9.3 K │ train │     0 │
│ 8  │ decoder_vsn                       │ _VariableSelectionNetwork        │  6.0 K │ train │     0 │
│ 9  │ static_context_grn                │ _GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_hidden_encoder_grn │ _GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ static_context_cell_encoder_grn   │ _GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 12 │ static_context_enrichment         │ _GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 13 │ lstm_encoder                      │ LSTM                             │ 16.9 K │ train │     0 │
│ 14 │ lstm_decoder                      │ LSTM                             │ 16.9 K │ train │     0 │
│ 15 │ post_lstm_gan                     │ _GateAddNorm                     │  2.2 K │ train │     0 │
│ 16 │ static_enrichment_grn             │ _GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 17 │ multihead_attn                    │ _InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 18 │ post_attn_gan                     │ _GateAddNorm                     │  2.2 K │ train │     0 │
│ 19 │ feed_forward_block                │ _GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 20 │ pre_output_gan                    │ _GateAddNorm                     │  2.2 K │ train │     0 │
│ 21 │ output_layer                      │ Linear                           │     33 │ train │     0 │
└────┴───────────────────────────────────┴──────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 85.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.0 K                                                                                               
Total estimated model params size (MB): 0.680                                                                      
Modules in train mode: 365                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/nuranisa/miniconda3/envs/ml_protel/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/home/nuranisa/miniconda3/envs/ml_protel/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: 
UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be 
used.
  super().__init__(loader)

`Trainer.fit` stopped: `max_epochs=5` reached.


TFTModel(output_chunk_shift=0, hidden_size=32, lstm_layers=2, num_attention_heads=4, full_attention=False, feed_forward=GatedResidualNetwork, dropout=0.1, hidden_continuous_size=8, categorical_embedding_sizes=None, add_relative_index=False, skip_interpolation=False, loss_fn=MSELoss(), likelihood=None, norm_type=LayerNorm, use_static_covariates=True, input_chunk_length=120, output_chunk_length=30, batch_size=64, n_epochs=5, optimizer_kwargs={'lr': 0.001}, add_encoders={'cyclic': {'future': ['minute', 'second', 'hour']}, 'transformer': Scaler}, pl_trainer_kwargs={'accelerator': 'auto', 'callbacks': [<pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint object at 0x7afd61a36120>], 'logger': <pytorch_lightning.loggers.csv_logs.CSVLogger object at 0x7afd629d4980>, 'enable_checkpointing': True, 'log_every_n_steps': 1}, random_state=42)

In [13]:
import pandas as pd

# 1. Baca file log mentah
metrics_df = pd.read_csv("logs/tft_run_1779506993.8042936/version_0/metrics.csv")

# 2. Kelompokkan berdasarkan 'epoch' dan hitung rata-ratanya
# Ini akan otomatis meratakan semua train_loss di epoch yang sama
epoch_metrics = metrics_df.groupby("epoch").agg({
    "train_loss": "mean",  # Mengambil rata-rata train_loss per epoch
    "val_loss": "first"    # val_loss sudah dicatat per epoch, ambil nilai yang ada
}).reset_index()

# 3. Hasilnya adalah tabel bersih per epoch
print("--- Metrics Per Epoch ---")
print(epoch_metrics)

--- Metrics Per Epoch ---
   epoch  train_loss  val_loss
0      0   12.026674  7.319647
1      1    6.937709  4.995517
2      2    4.838176  3.437774
3      3    3.423419  2.447186
4      4    2.492236  1.737182
